# Task 1: Design an In-Silico Perturbation Workflow

The idea here is to simulate graded gene knock-up and knock-down by scaling expression at the count level, then re-embedding through GeneFormer V2 to see how the embedding shifts.

**Key design choice**: GeneFormer's built-in perturbation only supports binary delete/overexpress. But real gene regulation is continuous; a 2-fold upregulation is very different from slamming a gene to rank 1. So instead, I scale the raw counts by a factor (0 = knockout through 10x = strong upregulation), which changes the gene's rank naturally and propagates through the tokenization.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.spatial.distance import cosine
from scipy import sparse
import torch
from copy import deepcopy
from tqdm import tqdm

from helical.models.geneformer import Geneformer, GeneformerConfig

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 1. Load and explore the dataset

In [ ]:
DATA_PATH = "../data/counts_combined_filtered_BA4_sALS_PN.h5ad"
adata = sc.read_h5ad(DATA_PATH)
print(f"Dataset: {adata.shape[0]} cells x {adata.shape[1]} genes")
print(f"\nCell metadata columns: {list(adata.obs.columns)}")
print(f"Gene metadata columns: {list(adata.var.columns)}")
print(f"\nadata.X type: {type(adata.X)}, dtype: {adata.X.dtype}")
if sparse.issparse(adata.X):
    print(f"Sparse format: {adata.X.format}")
    print(f"Sparsity: {1 - adata.X.nnz / np.prod(adata.X.shape):.2%}")

In [ ]:
# Explore cell metadata
print("=== Cell metadata summary ===")
for col in adata.obs.columns:
    n_unique = adata.obs[col].nunique()
    if n_unique <= 20:
        print(f"\n{col} ({n_unique} categories):")
        print(adata.obs[col].value_counts())
    else:
        print(f"\n{col}: {n_unique} unique values")

In [ ]:
# Identify disease status and cell type columns
# Column names identified from exploration
DISEASE_COL   = 'Condition'       # ALS vs PN
CELLTYPE_COL  = 'CellType'        # 19 broad types
SUBTYPE_COL   = 'SubType'         # 40 subtypes (includes SCN4B_NEFH — vulnerable neurons)
HEALTHY_LABEL = 'PN'              # post-mortem normal
DISEASE_LABEL = 'ALS'             # sporadic ALS

healthy_mask = (adata.obs[DISEASE_COL] == HEALTHY_LABEL).values
disease_mask = (adata.obs[DISEASE_COL] == DISEASE_LABEL).values

print(f"Condition: {DISEASE_COL}")
print(f"  Healthy ({HEALTHY_LABEL}): {healthy_mask.sum():,}")
print(f"  Disease ({DISEASE_LABEL}): {disease_mask.sum():,}")
print(f"\nCell types ({CELLTYPE_COL}): {adata.obs[CELLTYPE_COL].nunique()}")
print(f"Subtypes ({SUBTYPE_COL}): {adata.obs[SUBTYPE_COL].nunique()}")
print(f"\nVulnerable subtype SCN4B_NEFH: {(adata.obs[SUBTYPE_COL] == 'SCN4B_NEFH').sum():,} cells")

In [ ]:
# Basic QC metrics
adata.var['mt'] = adata.var_names.str.startswith('MT-')
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], inplace=True, log1p=True)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
sc.pl.violin(adata, 'n_genes_by_counts', ax=axes[0], show=False)
sc.pl.violin(adata, 'total_counts', ax=axes[1], show=False)
sc.pl.violin(adata, 'pct_counts_mt', ax=axes[2], show=False)
plt.tight_layout()
plt.savefig('../slides/task1_qc.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Median genes/cell: {np.median(adata.obs['n_genes_by_counts']):.0f}")
print(f"Median counts/cell: {np.median(adata.obs['total_counts']):.0f}")
print(f"Median MT%: {np.median(adata.obs['pct_counts_mt']):.1f}%")

## 1b. Scanpy exploratory analysis

Standard single-cell preprocessing to understand the data before perturbation. We keep raw counts in `adata.layers['counts']` for GeneFormer (which needs raw counts), and work with normalized data for visualization and DE.

In [ ]:
# Preserve raw counts for GeneFormer (needs unnormalized counts)
adata.layers['counts'] = adata.X.copy()

# Normalize + log-transform for scanpy analysis
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

# Highly variable genes
sc.pp.highly_variable_genes(adata, n_top_genes=3000, flavor='seurat_v3', layer='counts')
print(f"Highly variable genes: {adata.var['highly_variable'].sum()}")

# PCA on HVGs
adata_hvg = adata[:, adata.var['highly_variable']].copy()
sc.tl.pca(adata_hvg, n_comps=50)
adata.obsm['X_pca'] = adata_hvg.obsm['X_pca']

# Neighbors + UMAP + Leiden clustering
sc.pp.neighbors(adata, use_rep='X_pca', n_neighbors=30, n_pcs=30)
sc.tl.umap(adata, random_state=42)
sc.tl.leiden(adata, resolution=0.5, random_state=42)

print(f"Leiden clusters: {adata.obs['leiden'].nunique()}")

In [ ]:
# Visualize scanpy UMAP
fig, axes = plt.subplots(2, 2, figsize=(18, 14))

sc.pl.umap(adata, color='leiden', ax=axes[0, 0], show=False, title='Leiden clusters', legend_loc='on data')
sc.pl.umap(adata, color=DISEASE_COL, ax=axes[0, 1], show=False, title='Condition (ALS vs PN)')
sc.pl.umap(adata, color=CELLTYPE_COL, ax=axes[1, 0], show=False, title='Cell type (19 types)')
sc.pl.umap(adata, color='CellClass', ax=axes[1, 1], show=False, title='Cell class (Ex/Glia/In/Vasc)')

plt.tight_layout()
plt.savefig('../slides/task1_scanpy_umap.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Differential expression: ALS vs PN (healthy controls)
sc.tl.rank_genes_groups(adata, groupby=DISEASE_COL, groups=[DISEASE_LABEL], reference=HEALTHY_LABEL, method='wilcoxon')

de_df = sc.get.rank_genes_groups_df(adata, group=DISEASE_LABEL)
de_sig = de_df[de_df['pvals_adj'] < 0.05]
print(f"DE genes (FDR < 0.05): {len(de_sig)} / {len(de_df)}")
print(f"  Upregulated: {(de_sig['logfoldchanges'] > 0).sum()}")
print(f"  Downregulated: {(de_sig['logfoldchanges'] < 0).sum()}")
print(f"\nTop 15 upregulated in disease:")
print(de_sig.sort_values('logfoldchanges', ascending=False).head(15)[['names','logfoldchanges','pvals_adj']].to_string(index=False))
print(f"\nTop 15 downregulated in disease:")
print(de_sig.sort_values('logfoldchanges', ascending=True).head(15)[['names','logfoldchanges','pvals_adj']].to_string(index=False))

In [ ]:
# Volcano plot
fig, ax = plt.subplots(figsize=(10, 7))
de_df['neg_log10_padj'] = -np.log10(de_df['pvals_adj'] + 1e-300)

sig_up = de_df[(de_df['pvals_adj'] < 0.05) & (de_df['logfoldchanges'] > 0.5)]
sig_down = de_df[(de_df['pvals_adj'] < 0.05) & (de_df['logfoldchanges'] < -0.5)]
ns = de_df[~de_df.index.isin(sig_up.index) & ~de_df.index.isin(sig_down.index)]

ax.scatter(ns['logfoldchanges'], ns['neg_log10_padj'], s=3, alpha=0.3, c='gray', label='NS')
ax.scatter(sig_up['logfoldchanges'], sig_up['neg_log10_padj'], s=5, alpha=0.5, c='indianred', label=f'Up ({len(sig_up)})')
ax.scatter(sig_down['logfoldchanges'], sig_down['neg_log10_padj'], s=5, alpha=0.5, c='steelblue', label=f'Down ({len(sig_down)})')

# Label ALS genes
als_genes_check = ['TARDBP', 'SOD1', 'FUS', 'C9orf72', 'STMN2', 'ATXN2', 'UNC13A', 'POU3F1', 'SCN4B']
for gene in als_genes_check:
    match = de_df[de_df['names'] == gene]
    if len(match) > 0:
        row = match.iloc[0]
        ax.annotate(gene, (row['logfoldchanges'], row['neg_log10_padj']),
                   fontsize=8, fontweight='bold', color='black',
                   arrowprops=dict(arrowstyle='-', color='black', lw=0.5),
                   xytext=(5, 5), textcoords='offset points')

ax.set_xlabel('log2 fold change (disease vs healthy)', fontsize=12)
ax.set_ylabel('-log10(adjusted p-value)', fontsize=12)
ax.set_title('Differential expression: disease vs healthy', fontsize=13)
ax.axhline(-np.log10(0.05), color='gray', linestyle='--', alpha=0.5)
ax.legend()
plt.tight_layout()
plt.savefig('../slides/task1_volcano.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Restore raw counts for GeneFormer (it needs unnormalized counts)
adata.X = adata.layers['counts'].copy()
print("Restored raw counts to adata.X for GeneFormer embedding")

## 2. GeneFormer V2 embeddings

We use the 12-layer, 95M-corpus model (`gf-12L-95M-i4096`, renamed to `gf-12L-38M-i4096` in the Helical SDK) with 4096-gene input window. Baseline embeddings can be loaded from cache if already computed via `scripts/embed_baseline_parallel.py`, or computed here.

In [ ]:
import os

BASELINE_EMB_PATH = '../data/embeddings_baseline.npy'

if os.path.exists(BASELINE_EMB_PATH):
    print(f"Baseline embeddings found at {BASELINE_EMB_PATH} — loading from cache")
    embeddings_baseline = np.load(BASELINE_EMB_PATH)
    adata.obsm['X_geneformer'] = embeddings_baseline
    print(f"Loaded: {embeddings_baseline.shape}")
    EMBEDDINGS_CACHED = True
else:
    print("No cached embeddings found — will compute from scratch")
    EMBEDDINGS_CACHED = False

# Initialize GeneFormer (needed for demo perturbation even if embeddings are cached)
device = "cuda" if torch.cuda.is_available() else "cpu"
gf_config = GeneformerConfig(
    model_name="gf-12L-95M-i4096",
    device=device,
    batch_size=16,
)
geneformer = Geneformer(configurer=gf_config)
print(f"GeneFormer V2 loaded on {device}")

## 3. Compute baseline embeddings

Embed all cells in their unperturbed state. This serves as the reference for measuring perturbation effects.

For speed, use `scripts/embed_baseline_parallel.py` to distribute across 8 GPUs (~3 min). If already computed, this cell loads from cache.

In [ ]:
if not EMBEDDINGS_CACHED:
    print("Computing baseline embeddings for all cells...")
    dataset_baseline = geneformer.process_data(adata)
    embeddings_baseline = geneformer.get_embeddings(dataset_baseline)
    adata.obsm['X_geneformer'] = embeddings_baseline
    np.save(BASELINE_EMB_PATH, embeddings_baseline)
    print(f"Saved to {BASELINE_EMB_PATH}")
else:
    print(f"Using cached embeddings: {embeddings_baseline.shape}")

print(f"Baseline embeddings: {embeddings_baseline.shape[0]} cells × {embeddings_baseline.shape[1]} dims")

In [ ]:
# UMAP of GeneFormer embeddings
import umap.umap_ as umap

reducer = umap.UMAP(n_neighbors=30, min_dist=0.3, random_state=42)
umap_baseline = reducer.fit_transform(embeddings_baseline)
adata.obsm['X_umap_gf'] = umap_baseline

fig, axes = plt.subplots(1, 3, figsize=(22, 6))
sc.pl.embedding(adata, basis='umap_gf', color=DISEASE_COL, ax=axes[0], show=False, title='GeneFormer: Condition')
sc.pl.embedding(adata, basis='umap_gf', color=CELLTYPE_COL, ax=axes[1], show=False, title='GeneFormer: Cell type')
sc.pl.embedding(adata, basis='umap_gf', color=SUBTYPE_COL, ax=axes[2], show=False, title='GeneFormer: Subtype')
plt.tight_layout()
plt.savefig('../slides/task1_geneformer_umap.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Perturbation workflow

### Design

GeneFormer tokenizes cells by **rank-ordering genes by expression** (highest expressed gene = first token). Our perturbation strategy operates at the expression level:

- **Knock-down**: multiply target gene expression by a factor < 1 (e.g., 0.5, 0.25, 0.0)
- **Knock-up**: multiply target gene expression by a factor > 1 (e.g., 2, 5, 10)

This changes the gene's rank relative to all other genes, which propagates through GeneFormer's tokenization. A gene knocked down from the top 5% to the bottom 50% will dramatically shift the token sequence. Multiple dose levels give us a dose-response curve in embedding space.

**Why this is better than binary delete/overexpress**: Biological regulation is graded, not binary. A 2-fold upregulation of TARDBP is different from moving it to rank 1. Our approach captures this continuum.

In [ ]:
# Perturbation dose levels
DOSE_LEVELS = {
    'knockout':    0.0,    # complete loss
    'strong_down': 0.1,    # 90% reduction
    'moderate_down': 0.5,  # 50% reduction
    'mild_down':   0.75,   # 25% reduction
    'mild_up':     1.5,    # 50% increase
    'moderate_up': 2.0,    # 2-fold increase
    'strong_up':   5.0,    # 5-fold increase
    'extreme_up':  10.0,   # 10-fold increase
}


def perturb_gene(adata, gene_name, factor):
    """Create a perturbed copy of adata by scaling a gene's expression.
    
    Multiplies the target gene's raw counts by `factor` in every cell.
    The rest of the transcriptome is unchanged, but the rank ordering
    (which GeneFormer uses for tokenization) will shift.
    
    Parameters
    ----------
    adata : AnnData
        Original (unperturbed) data with raw counts in .X
    gene_name : str
        Gene symbol to perturb
    factor : float
        Scaling factor. 0 = knockout, <1 = knockdown, >1 = knockup
        
    Returns
    -------
    AnnData
        Copy with perturbed expression for the target gene
    """
    if gene_name not in adata.var_names:
        raise ValueError(f"Gene '{gene_name}' not found in dataset. Available: {adata.var_names[:5].tolist()}...")
    
    adata_pert = adata.copy()
    gene_idx = list(adata_pert.var_names).index(gene_name)
    
    if sparse.issparse(adata_pert.X):
        X = adata_pert.X.tocsc()
        col = X.getcol(gene_idx).toarray().flatten() * factor
        col = np.round(col).astype(X.dtype)
        X_lil = X.tolil()
        X_lil[:, gene_idx] = col.reshape(-1, 1)
        adata_pert.X = X_lil.tocsr()
    else:
        adata_pert.X[:, gene_idx] = np.round(adata_pert.X[:, gene_idx] * factor).astype(adata.X.dtype)
    
    return adata_pert


def embed_perturbation(geneformer, adata_perturbed):
    """Embed perturbed cells using GeneFormer."""
    dataset = geneformer.process_data(adata_perturbed)
    embeddings = geneformer.get_embeddings(dataset)
    return embeddings


def compute_perturbation_effect(emb_baseline, emb_perturbed):
    """Compute per-cell cosine distance between baseline and perturbed embeddings."""
    # Normalize for cosine similarity
    norm_base = emb_baseline / (np.linalg.norm(emb_baseline, axis=1, keepdims=True) + 1e-8)
    norm_pert = emb_perturbed / (np.linalg.norm(emb_perturbed, axis=1, keepdims=True) + 1e-8)
    
    # Cosine distance = 1 - cosine similarity
    cosine_sim = np.sum(norm_base * norm_pert, axis=1)
    cosine_dist = 1 - cosine_sim
    
    # Also compute the shift vector (direction of perturbation effect)
    shift_vectors = emb_perturbed - emb_baseline
    shift_magnitude = np.linalg.norm(shift_vectors, axis=1)
    
    return {
        'cosine_distance': cosine_dist,
        'shift_magnitude': shift_magnitude,
        'shift_vectors': shift_vectors,
    }

## 5. Full perturbation pipeline

Wraps the above into a single function that takes a gene list and dose levels, runs all perturbations, and returns structured results.

In [ ]:
def run_perturbation_experiment(
    geneformer,
    adata,
    emb_baseline,
    gene_list,
    dose_levels=None,
    cell_mask=None,
):
    """Run a full perturbation experiment across genes and dose levels.
    
    Parameters
    ----------
    geneformer : Geneformer
        Initialized GeneFormer model
    adata : AnnData
        Dataset with raw counts
    emb_baseline : np.ndarray
        Baseline embeddings for all cells in adata
    gene_list : list of str
        Genes to perturb
    dose_levels : dict, optional
        {name: factor} mapping. Default: DOSE_LEVELS
    cell_mask : np.ndarray of bool, optional
        If provided, only perturb and analyze this subset of cells.
        Embeddings are still computed for all cells but effects
        are only recorded for the masked subset.
    
    Returns
    -------
    dict
        Nested: results[gene][dose] = {
            'cosine_distance': array,
            'shift_magnitude': array,
            'shift_vectors': array,
            'embeddings': array,
        }
    """
    if dose_levels is None:
        dose_levels = DOSE_LEVELS
    
    # Subset adata if mask provided
    if cell_mask is not None:
        adata_sub = adata[cell_mask].copy()
        emb_base_sub = emb_baseline[cell_mask]
    else:
        adata_sub = adata
        emb_base_sub = emb_baseline
    
    results = {}
    available_genes = [g for g in gene_list if g in adata_sub.var_names]
    missing_genes = [g for g in gene_list if g not in adata_sub.var_names]
    
    if missing_genes:
        print(f"Warning: genes not found in dataset: {missing_genes}")
    
    for gene in tqdm(available_genes, desc="Genes"):
        results[gene] = {}
        for dose_name, factor in dose_levels.items():
            # Perturb
            adata_pert = perturb_gene(adata_sub, gene, factor)
            
            # Embed
            emb_pert = embed_perturbation(geneformer, adata_pert)
            
            # Compute effect
            effect = compute_perturbation_effect(emb_base_sub, emb_pert)
            effect['embeddings'] = emb_pert
            effect['factor'] = factor
            
            results[gene][dose_name] = effect
            
            print(f"  {gene} | {dose_name} (x{factor}): "
                  f"mean cosine dist = {effect['cosine_distance'].mean():.4f}, "
                  f"mean shift = {effect['shift_magnitude'].mean():.4f}")
    
    return results

## 6. Demo: perturb a single gene to validate the workflow

In [ ]:
# Pick a well-known ALS gene to test the workflow
# TARDBP (TDP-43) is present in 97% of ALS cases
TEST_GENE = 'TARDBP'

if TEST_GENE in adata.var_names:
    print(f"{TEST_GENE} found in dataset")
    gene_idx = list(adata.var_names).index(TEST_GENE)
    if sparse.issparse(adata.X):
        expr = adata.X[:, gene_idx].toarray().flatten()
    else:
        expr = adata.X[:, gene_idx].flatten()
    print(f"  Expression: mean={expr.mean():.2f}, median={np.median(expr):.2f}, "
          f"nonzero={np.sum(expr > 0)}/{len(expr)} ({np.mean(expr > 0):.1%})")
else:
    print(f"{TEST_GENE} not found. Checking alternatives...")
    for alt in ['SOD1', 'FUS', 'C9orf72', 'STMN2']:
        if alt in adata.var_names:
            TEST_GENE = alt
            print(f"Using {TEST_GENE} instead")
            break

In [ ]:
# Run a quick test with knockout and strong upregulation on a small subset
N_TEST = min(2000, adata.shape[0])
test_mask = np.zeros(adata.shape[0], dtype=bool)
test_mask[np.random.RandomState(42).choice(adata.shape[0], N_TEST, replace=False)] = True

test_doses = {
    'knockout': 0.0,
    'strong_down': 0.1,
    'baseline': 1.0,  # no change, sanity check
    'strong_up': 5.0,
    'extreme_up': 10.0,
}

test_results = run_perturbation_experiment(
    geneformer=geneformer,
    adata=adata,
    emb_baseline=embeddings_baseline,
    gene_list=[TEST_GENE],
    dose_levels=test_doses,
    cell_mask=test_mask,
)

In [ ]:
# Sanity check: baseline (factor=1.0) should have ~0 shift
baseline_shift = test_results[TEST_GENE]['baseline']['cosine_distance'].mean()
print(f"Sanity check — factor=1.0 mean cosine distance: {baseline_shift:.6f}")
assert baseline_shift < 0.01, f"Baseline shift too large ({baseline_shift:.4f}), something is wrong"
print("PASSED: no-perturbation control produces near-zero shift")

In [ ]:
# Plot dose-response curve
doses = []
mean_shifts = []
for dose_name, effect in test_results[TEST_GENE].items():
    if dose_name == 'baseline':
        continue
    doses.append(effect['factor'])
    mean_shifts.append(effect['cosine_distance'].mean())

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(doses, mean_shifts, 'o-', markersize=8, linewidth=2)
ax.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
ax.axvline(x=1.0, color='red', linestyle='--', alpha=0.5, label='No perturbation')
ax.set_xlabel('Expression scaling factor', fontsize=12)
ax.set_ylabel('Mean cosine distance from baseline', fontsize=12)
ax.set_title(f'Dose-response: {TEST_GENE} perturbation in GeneFormer embedding space', fontsize=13)
ax.set_xscale('log')
ax.legend()
plt.tight_layout()
plt.savefig('../slides/task1_dose_response.png', dpi=150, bbox_inches='tight')
plt.show()

## Summary

The perturbation workflow:

1. **Takes** an AnnData with raw counts, a list of target genes, and dose levels
2. **Perturbs** each gene by scaling its expression (0 = knockout → 10x = strong upregulation)
3. **Re-embeds** perturbed cells through GeneFormer V2, which re-ranks all genes based on the modified expression
4. **Quantifies** perturbation effects as cosine distance and shift vectors in embedding space
5. **Validates** with a sanity check (factor=1.0 → zero shift) and dose-response curve

The workflow scales to any number of genes via `run_perturbation_experiment()`. In Task 2 we apply it to ALS disease genes.